# Testing the performance of VGG16

In [1]:
# import cv2
# import os

# input_folder = "./data/europeana_videos_stills"
# output_folder = "./data/europeana_videos_stills"

# os.makedirs(output_folder, exist_ok=True)  # Ensure output directory exists

# for filename in os.listdir(input_folder):
#     if filename.endswith(".png"):  
#         img_path = os.path.join(input_folder, filename)
        
#         # Read the image (forcefully add an alpha channel if missing)
#         img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        
#         if img.shape[2] == 3:  # If no alpha channel, add one
#             b, g, r = cv2.split(img)
#             a = np.ones_like(b) * 255  # Create a full white alpha channel
#             img = cv2.merge((b, g, r, a))

#         # Save image
#         cv2.imwrite(os.path.join(output_folder, filename), img)
#         print(f"Converted {filename} to RGBA.")

In [1]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications.vgg16 import decode_predictions
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.models import Model
from tensorflow import keras
import tensorflow as tf
import os
import numpy as np

2025-03-11 08:27:53.007216: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-11 08:27:53.016204: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741678073.026680   20396 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741678073.029634   20396 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-11 08:27:53.041314: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
tf.config.set_visible_devices([], 'GPU')

2025-03-11 08:27:57.312542: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
def image_files(path):
    '''
    Creates a list with the names of the images to be processed.
    '''
    image_files = []
    # creates a ScandirIterator aliased as files
    with os.scandir(path) as files:
        for file in files:
            if file.name.endswith('.jpg'):
                # adds only the image files to the list
                image_files.append(file.name)
    return sorted(image_files)


def load_model():
    model = VGG16()
    model = Model(inputs = model.inputs, outputs = model.output)
    return model


def recognize_objects(file, path, model):
    '''
    Recognizes objects in an image file using the VGG16 model.
    '''
    # load the image as a 224x224 array
    img = load_img(f'{path}/{file}', target_size=(224,224))
    # convert from 'PIL.Image.Image' to numpy array
    img = np.array(img, dtype=np.float32)
    
    reshaped_img = img.reshape(1,224,224,3)
    # prepare image for model
    imgx = preprocess_input(reshaped_img)
    # get the feature vector
    objects = model.predict(imgx)

    return objects

In [4]:
path = "./data/europeana_videos_stills"
model = load_model()
#print(model.input_shape)  # Expected input shape

files = image_files(path)

for file in files: 
    preds= recognize_objects(file, path, model)
    top_preds = decode_predictions(preds, top=5)[0]

    print(f"Objects detected in {file} are:")
    for i, (imagenet_id, label, score) in enumerate(top_preds):
        print(f"{i+1}. {label} ({score*100:.2f}%)")
    print()
       

/home/rallypal/.pyenv/versions/lewagon/lib/python3.10/site-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
Objects detected in 01.jpg are:
1. paddlewheel (21.80%)
2. crane (16.95%)
3. pier (11.50%)
4. dock (6.11%)
5. steel_arch_bridge (5.66%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
Objects detected in 02.jpg are:
1. thresher (59.15%)
2. lumbermill (19.23%)
3. tractor (6.01%)
4. oxcart (4.35%)
5. horse_cart (3.64%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
Objects detected in 03.jpg are:
1. shovel (4.12%)
2. fountain (3.76%)
3. paintbrush (3.59%)
4. stretcher (3.14%)
5. dogsled (3.08%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
Objects detected in 04.jpg are:
1. missile (26.10%)
2. projectile (24.86%)
3. cannon (24.01%)
4. tank (4.28%)
5. military_uniform (2.33%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
Objects detected in 05.jpg are:
1. guillotine (22.96%)
2. paintbrush (3.88%)
3. quill (3.47%)
4. scale (2.41%)
5. butcher_shop (2.29%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
Objects detected in 06.jpg are:
1. stage (10.23%)
2. book_jacket (5.67%)
3. abaya (4.

# Testing the performance of CLIP:

In [6]:
import os
import torch
import clip
from PIL import Image
from pathlib import Path

# Load the CLIP model and preprocessing function
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

prompt= 'outdoor scene'
text = clip.tokenize([prompt]).to(device)

# Path to the image directory
image_dir = Path("./data/europeana_videos_stills")

my_images = []

for image_path in sorted(image_dir.glob("*.jpg")):  
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    # Compute image features
    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text)

    # Compute similarity score between the image and the "car" prompt
    similarity = image_features @ text_features.T

    # The threshold is the key!
    threshold = 23.0 
    if similarity.item() > threshold:
        my_images.append(image_path)

print(f"Images that display {prompt}:")
for each_image in my_images:
    print(each_image)

Images that display outdoor scene:
data/europeana_videos_stills/01.jpg
data/europeana_videos_stills/06.jpg
data/europeana_videos_stills/08.jpg
